In [2]:
import sys
sys.path.insert(0, "..")  # Find numcompute in the parent folder.

sample_path = "data/sample.csv"

## 1. Read CSV Data

Skip the header and fill missing values with `np.nan`. The result is a two-dimensional NumPy array. Other values remain strings, preserving city names.

In [3]:
from numcompute.io import load_csv

X = load_csv(sample_path, delimiter=",", skip_header=1)
print(X.shape)
print(X[:10])

(1200, 3)
[['20' '30000' 'Adelaide']
 [nan '40000' 'Melbourne']
 ['40' nan 'Adelaide']
 ['30' '50000' 'Sydney']
 ['25' '35000' nan]
 ['63' '122750.65' 'Adelaide']
 ['22' '133350.18' 'Melbourne']
 ['54' '39263.01' 'Perth']
 ['28' '116551.69' 'Brisbane']
 ['22' '80011.09' nan]]


You can also skip rows containing missing values:

In [4]:
X_clean = load_csv(sample_path, skip_header=1, missing="skip")
print(X_clean.shape)

(982, 3)


For tab-separated files, use `delimiter="\t"`.

## 2. Preprocessing

Each class provides `fit`, `transform`, and `fit_transform`. `fit` learns from the input and returns the object; `transform` applies the learned settings. Inputs have shape `(samples, features)`.

### 2.1 Fill missing values

Convert age and income to floats, then replace NaNs with a constant. Zero is used here only to demonstrate constant filling.

In [5]:
import numpy as np
from numcompute.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, Imputer

X_numeric = X[:, :2].astype(float)
imputer = Imputer(fill_value = 0)
X_filled = imputer.fit_transform(X_numeric)
print(X_filled[:5])

[[2.0e+01 3.0e+04]
 [0.0e+00 4.0e+04]
 [4.0e+01 0.0e+00]
 [3.0e+01 5.0e+04]
 [2.5e+01 3.5e+04]]


### 2.2 StandardScaler

Standardize each column: `(X - mean) / std`. The training columns have mean approximately 0 and population standard deviation 1. Constant columns become 0.

In [6]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filled)
print(X_scaled[:5])
print("Column means:", np.round(X_scaled.mean(axis=0), 6))
print("Column standard deviations:", np.round(X_scaled.std(axis=0), 6))

[[-1.19409505 -1.1934783 ]
 [-2.2966759  -0.95677933]
 [-0.09151421 -1.90357521]
 [-0.64280463 -0.72008036]
 [-0.91844984 -1.07512881]]
Column means: [ 0. -0.]
Column standard deviations: [1. 1.]


Use `transform` on new data to reuse the training mean and standard deviation. Do not call `fit` again on the new data.

In [7]:
X_new = np.array([[25, 35000], [30, 50000]], dtype=float)
print(scaler.transform(X_new))

[[-0.91844984 -1.07512881]
 [-0.64280463 -0.72008036]]


### 2.3 MinMaxScaler

Scale each training column to a given range. The default is `(0, 1)`; here we use `(-1, 1)`. Constant columns map to the lower bound. New data is not clipped.

In [8]:
minmax = MinMaxScaler(feature_range=(-1, 1))
X_minmax = minmax.fit_transform(X_filled)
print(X_minmax[:5])
print("Column minima:", X_minmax.min(axis=0))
print("Column maxima:", X_minmax.max(axis=0))

[[-0.42857143 -0.59980721]
 [-1.         -0.46640962]
 [ 0.14285714 -1.        ]
 [-0.14285714 -0.33301202]
 [-0.28571429 -0.53310842]]
Column minima: [-1. -1.]
Column maxima: [1. 1.]


### 2.4 OneHotEncoder

Fill missing cities with `Unknown`, then create one column per category. Categories are sorted. Unseen categories produce all zeros. For multiple categorical input columns, each column has its own category group.

In [9]:
cities = X[:, 2:3]
city_imputer = Imputer(fill_value="Unknown")
cities_filled = city_imputer.fit_transform(cities)

encoder = OneHotEncoder()
X_encoded = encoder.fit_transform(cities_filled)
print("Categories:", encoder.categories_)
print("Shape:", X_encoded.shape)
print(X_encoded[:5])

Categories: ['Adelaide' 'Brisbane' 'Hobart' 'Melbourne' 'Perth' 'Sydney' 'Unknown']
Shape: (1200, 7)
[[1 0 0 0 0 0 0]
 [0 0 0 1 0 0 0]
 [1 0 0 0 0 0 0]
 [0 0 0 0 0 1 0]
 [0 0 0 0 0 0 1]]


Both scalers ignore NaNs during `fit` and preserve them during `transform`. Fill an entirely missing column before fitting. OneHotEncoder requires missing categories to be filled first.

## 3. Pipeline

A Pipeline runs named steps in order. Here we use the same CSV data loaded as `X`. Each step receives the output of the previous step.

### 3.1 Fill and standardize numerical columns

Use the first 900 rows to fit the pipeline. Fill missing age/income values with 0, then standardize each column. Zero filling is used only as a simple example.

In [10]:
from numcompute.pipeline import Pipeline

numeric_input = X[:, :2].astype(float)
X_train = numeric_input[:900]
X_test = numeric_input[900:]

numeric_pipe = Pipeline([
    ("impute", Imputer(fill_value=0)),
    ("scale", StandardScaler()),
])

X_train_ready = numeric_pipe.fit_transform(X_train)
print("Training shape:", X_train_ready.shape)
print(X_train_ready[:5])
print("Training column means:", np.round(X_train_ready.mean(axis=0), 6))

Training shape: (900, 2)
[[-1.208934   -1.21086601]
 [-2.31399251 -0.97177194]
 [-0.1038755  -1.92814824]
 [-0.65640475 -0.73267786]
 [-0.93266938 -1.09131897]]
Training column means: [-0.  0.]


### 3.2 Reuse the fitted pipeline

Call `transform` on the remaining 300 rows. This reuses the parameters learned from the first 900 rows and does not fit again.

In [10]:
X_test_ready = numeric_pipe.transform(X_test)
print("Test shape:", X_test_ready.shape)
print(X_test_ready[:5])

Test shape: (300, 2)
[[-2.31399251  0.67754133]
 [-2.31399251  1.54047011]
 [-0.71165768 -0.22187632]
 [ 1.22219471 -0.05998141]
 [ 0.00663035 -1.92814824]]


### 3.3 Fill and encode cities

For the categorical column, replace NaN with `Unknown`, then apply one-hot encoding.

In [11]:
city_pipe = Pipeline([
    ("impute", Imputer(fill_value="Unknown")),
    ("encode", OneHotEncoder()),
])

cities_ready = city_pipe.fit_transform(X[:, 2:3])
print("Encoded city shape:", cities_ready.shape)
print(cities_ready[:5])

Encoded city shape: (1200, 7)
[[1 0 0 0 0 0 0]
 [0 0 0 1 0 0 0]
 [1 0 0 0 0 0 0]
 [0 0 0 0 0 1 0]
 [0 0 0 0 0 0 1]]


`fit(X)` returns the pipeline itself; `transform(X)` returns the processed array. `fit_transform(X)` combines both operations. If the final step is a model providing `fit` and `predict`, the pipeline also supports `fit(X, y)` and `predict(X)`. Implementing or training a model is not required for this assignment.

## 8. Finite-Difference Gradients and Jacobians

Estimate derivatives by slightly changing each input coordinate. Forward differences compare `f(x+h)` with `f(x)`; central differences compare `f(x+h)` with `f(x-h)`.

Use the first standardized numerical row from the CSV pipeline as the evaluation point. This keeps age and income on comparable scales.

In [25]:
from numcompute.optim import grad, jacobian

x = X_train_ready[0].copy()
print("Evaluation point:", x)

Evaluation point: [-1.208934   -1.21086601]


### 8.1 Gradient of a scalar function

For `f(x) = sum(x**2)`, the analytical gradient is `2*x`. The result has the same shape as the one-dimensional input.

In [26]:
def f(x):
    return np.sum(x ** 2)

central_gradient = grad(f, x, h=1e-5, method="central")
forward_gradient = grad(f, x, h=1e-5, method="forward")
exact_gradient = 2 * x

print("Central gradient:", central_gradient)
print("Forward gradient:", forward_gradient)
print("Analytical gradient:", exact_gradient)
print("Central result matches:", np.allclose(central_gradient, exact_gradient))

Central gradient: [-2.41786801 -2.42173203]
Forward gradient: [-2.41785801 -2.42172203]
Analytical gradient: [-2.41786801 -2.42173203]
Central result matches: True


### 8.2 Jacobian of a vector function

For `F(x) = [x[0]**2 + x[1], x[0]*x[1]]`, the analytical Jacobian is `[[2*x[0], 1], [x[1], x[0]]]`. Rows correspond to output components and columns to input coordinates.

In [27]:
def F(x):
    return np.array([
        x[0] ** 2 + x[1],
        x[0] * x[1],
    ])

central_J = jacobian(F, x, method="central")
forward_J = jacobian(F, x, method="forward")
exact_J = np.array([[2 * x[0], 1], [x[1], x[0]]])

print("Central Jacobian:")
print(central_J)
print("Forward Jacobian:")
print(forward_J)
print("Analytical Jacobian:")
print(exact_J)
print("Shape:", central_J.shape)
print("Central result matches:", np.allclose(central_J, exact_J))

Central Jacobian:
[[-2.41786801  1.        ]
 [-1.21086601 -1.208934  ]]
Forward Jacobian:
[[-2.41785801  1.        ]
 [-1.21086601 -1.208934  ]]
Analytical Jacobian:
[[-2.41786801  1.        ]
 [-1.21086601 -1.208934  ]]
Shape: (2, 2)
Central result matches: True


`h` must be positive. A step that is too small can be lost to floating-point rounding. Functions must return finite values with the same shape at all evaluation points. These functions estimate derivatives; they do not train a model or update parameters.

## 9. Reproducible Benchmarks

To do: use a fixed seed, check result consistency, warm up, and repeat timings. Include measured performance results and environment information.